# ЛР2. Пункт 8. Коэффициент детерминации

Основной результат для отчёта — $R^2$ на выборке, на которой оценена модель.
Отложенные 300 CPU дают дополнительную оценку качества прогнозов.

## Воспроизведение модели ЛР1

Из `all_data_processed.csv` берём 1365 строк с меткой `split=train`. Оцениваем МНК по этим строкам. Совпадение прогнозов проверяем по `predictions.csv`.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


data_dir = Path(".") if Path("predictions.csv").is_file() else Path("lab2")

data = pd.read_csv(data_dir / "all_data_processed.csv")
training = data.loc[data["split"].eq("train")]
feature_names = [
    "log1p_cores", "base_clock_ghz", "log1p_tdp", "log1p_l3",
    "age_centered", "manufacturer_Intel", "manufacturer_Other", "category_Laptop",
]
X = np.column_stack([np.ones(len(training)), training[feature_names]])
y = training["ln_cpu_mark"].to_numpy()
coefficients = np.linalg.lstsq(X, y, rcond=None)[0]
y_hat = X @ coefficients

saved_predictions = pd.read_csv(data_dir / "predictions.csv")["y_hat"]
np.testing.assert_allclose(y_hat, saved_predictions, rtol=0, atol=1e-10)

## Расчёт по формуле лекции 3, слайд 14

$$
R^2=1-\frac{\mathrm{RSS}}{\mathrm{TSS}},\qquad
\mathrm{RSS}=\sum_i(y_i-\hat y_i)^2,\quad
\mathrm{TSS}=\sum_i(y_i-\bar y)^2.
$$

В сумму входит каждый процессор, включая наблюдения с совпадающими факторами.
Среднее $\bar y$ вычисляем отдельно для каждой выборки.

In [2]:
def determination(y_actual, predicted):
    rss = np.sum((y_actual - predicted) ** 2)
    tss = np.sum((y_actual - y_actual.mean()) ** 2)
    return {"n": len(y_actual), "RSS": rss, "TSS": tss, "R²": 1 - rss / tss}

test = data.loc[data["split"].eq("test")]
y_test = test["ln_cpu_mark"].to_numpy()
X_test = np.column_stack([np.ones(len(test)), test[feature_names]])
results = pd.DataFrame({
    "Обучающая": determination(y, y_hat),
    "Отложенная": determination(y_test, X_test @ coefficients),
}).T.astype({"n": int})

np.testing.assert_allclose(results["R²"], [0.884312, 0.873735], rtol=0, atol=5e-7)
display(results.round(6))

r2_dev, r2_test = results["R²"]
print(f"Модель описывает {r2_dev:.2%} разброса ln(CPU Mark) на обучающей выборке.")
print(f"На отложенной выборке R² = {r2_test:.4f}.")

Модель описывает 88.43% разброса ln(CPU Mark) на обучающей выборке.
На отложенной выборке R² = 0.8737.


,n,RSS,TSS,R²
Обучающая,1365,222.884528,1926.593641,0.884312
Отложенная,300,56.448497,447.062914,0.873735
